In [1]:
# from google.colab import drive
# drive.mount('/content/drive')

In [2]:
import os
os.chdir("/content")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [3]:
# !rm -rf "/content/vlm-safety-reasoning"

In [4]:
import subprocess
import shutil

DRIVE_ROOT = "/content/drive/MyDrive/vlm-finetuning-project1"
REPO_DIR = "vlm-safety-reasoning"
ENV_PATH = f"{DRIVE_ROOT}/secrets/.env"

def load_secrets(env_path: str) -> dict:
    """Read a .env file and export its values into os.environ."""
    if not os.path.exists(env_path):
        raise FileNotFoundError(f"Secrets file not found at: {env_path}")

    secrets = {}
    with open(env_path, "r") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, value = line.split("=", 1)
            secrets[key] = value.strip(" \"'\r")
            os.environ[key] = secrets[key]
    return secrets


print(">>> Loading secrets...")
secrets = load_secrets(ENV_PATH)
required_keys = ["GIT_EMAIL", "GIT_NAME", "GITHUB_USERNAME", "GITHUB_TOKEN", "HF_TOKEN"]
missing = [k for k in required_keys if k not in secrets]
if missing:
    raise KeyError(f"Missing required secrets: {missing}")
print(">>> Secrets loaded successfully.")

print(">>> Configuring Git identity...")
subprocess.run(["git", "config", "--global", "user.email", secrets["GIT_EMAIL"]], check=True)
subprocess.run(["git", "config", "--global", "user.name", secrets["GIT_NAME"]], check=True)

AUTH_REPO_URL = (
    f"https://{secrets['GITHUB_USERNAME']}:{secrets['GITHUB_TOKEN']}"
    f"@github.com/epmresearch/vlm-safety-reasoning.git"
)

if os.path.exists(REPO_DIR):
    print(">>> Repo already present, pulling latest...")
    os.chdir(REPO_DIR)
    subprocess.run(["git", "remote", "set-url", "origin", AUTH_REPO_URL], check=True)
    subprocess.run(["git", "pull", "origin", "main"], check=True)
else:
    print(">>> Cloning repo...")
    subprocess.run(["git", "clone", AUTH_REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

print(f">>> Working directory: {os.getcwd()}")

print(">>> Copying .env into local workspace...")
shutil.copy(ENV_PATH, ".env")

print(">>> Installing requirements...")
subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"], check=True)
print(">>> Setup complete.")

>>> Loading secrets...
>>> Secrets loaded successfully.
>>> Configuring Git identity...
>>> Cloning repo...
>>> Working directory: /content/vlm-safety-reasoning
>>> Copying .env into local workspace...
>>> Installing requirements...
>>> Setup complete.


In [5]:
from huggingface_hub import login
from pathlib import Path
import json

from core.config import load_config, load_task_config
from core.io import get_drive_path, ensure_dir
from core.run_manifest import save_run_manifest
from data.loader import load_processed_dataset
from models.model_loader import load_model_for_inference
from models.inference import run_inference_batched
from data.prompt_templates import SYSTEM_PROMPT, UNIFIED_INSPECTION_PROMPT

login(token=os.environ["HF_TOKEN"])

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [6]:
MODEL_TIER = "2b"
VARIANT = f"{MODEL_TIER}-baseline-p1"
N_SAMPLES = None # full test set
BATCH_SIZE = 80

# Setup Paths
DRIVE_RESULTS_DIR = get_drive_path("results", "inference", f"{VARIANT}-test")
ensure_dir(DRIVE_RESULTS_DIR)

JSONL_OUTPUT_PATH = str(DRIVE_RESULTS_DIR / "predictions.jsonl")
ADAPTER_PATH = None

# Load global configurations
base_config = load_config(training_kind="sft")
task_config = load_task_config("unified")

# Bundle the entire state to guarantee 100% reproducibility
run_config = {
    "experiment": f"inference_{VARIANT}_full_testset",
    "model_tier": MODEL_TIER,
    "notebook": "06_baseline_inference_2b_penalty_1.05.ipynb",
    "variant": VARIANT,
    "note": "This is the BASE model inference. Penalty 1.0 (off).",
    "adapter_path": ADAPTER_PATH,
    "batch_size": BATCH_SIZE,
    "n_samples": N_SAMPLES,
    "max_new_tokens": base_config.get("max_new_tokens", task_config.get("max_new_tokens", 1000)),
    "prompts": {
        "system_prompt": SYSTEM_PROMPT,
        "user_prompt": UNIFIED_INSPECTION_PROMPT
    },
    "full_base_and_sft_yaml_state": base_config,
    "full_task_yaml_state": task_config
}

# Save manifest to Drive
save_run_manifest(str(DRIVE_RESULTS_DIR), run_config)
print(f"Manifest saved. Results will stream to: {DRIVE_RESULTS_DIR}")

2026-07-27 20:30:25 | INFO     | core.run_manifest:save_run_manifest:38 - Run manifest saved to /content/drive/MyDrive/vlm-finetuning-project1/results/inference/2b-baseline-p1-test/run_manifest.json
Manifest saved. Results will stream to: /content/drive/MyDrive/vlm-finetuning-project1/results/inference/2b-baseline-p1-test


In [7]:
print("Loading fully processed dataset...")
splits = load_processed_dataset()
test_data = splits["test"]

if N_SAMPLES is not None:
    test_data = test_data.select(range(N_SAMPLES))

print(len(test_data), "samples loaded")

Loading fully processed dataset...
2026-07-27 20:30:55 | INFO     | data.loader:load_processed_dataset:183 - Loading fully processed dataset from disk: /content/drive/MyDrive/vlm-finetuning-project1/datasets/processed
2026-07-27 20:31:17 | INFO     | data.loader:load_processed_dataset:187 - Loaded processed 'train' split: 6308 samples
2026-07-27 20:31:17 | INFO     | data.loader:load_processed_dataset:187 - Loaded processed 'val' split: 701 samples
2026-07-27 20:31:17 | INFO     | data.loader:load_processed_dataset:187 - Loaded processed 'test' split: 3004 samples
3004 samples loaded


In [8]:
print(f"\nLoading base model (Tier: {MODEL_TIER})...")
model, tokenizer, info = load_model_for_inference(
    tier=MODEL_TIER,
    adapter_path=None
)
print("Base model loaded successfully!")


Loading base model (Tier: 2b)...
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
2026-07-27 20:31:45 | INFO     | models.model_loader:load_model_for_inference:207 - Loading model for inference: unsloth/Qwen3-VL-2B-Instruct with max_seq_length=2816
==((====))==  Unsloth 2026.7.5: Fast Qwen3_Vl patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

2026-07-27 20:32:10 | INFO     | models.model_loader:apply_pixel_bounds:255 - Applied image pixel bounds via size dict: min=200704, max=1204224
Base model loaded successfully!


In [9]:
# Run Inference
print(f"Starting batched inference for Base Model on {len(test_data)} samples...")
results_base = run_inference_batched(
    model=model,
    tokenizer=tokenizer,
    dataset=test_data,
    batch_size=run_config["batch_size"],
    max_new_tokens=run_config["max_new_tokens"],
    max_samples=run_config["n_samples"],
    output_path=JSONL_OUTPUT_PATH,
)

print(f"\nBase Model Inference Complete! {len(results_base)} total samples processed.")
print(f"Base predictions are secured in: {JSONL_OUTPUT_PATH}")

Starting batched inference for Base Model on 3004 samples...


Batched Inference: 100%|██████████| 38/38 [1:33:27<00:00, 147.55s/it]

2026-07-27 22:05:37 | INFO     | models.inference:run_inference_batched:348 - Batched inference complete: 3004 new samples processed.
2026-07-27 22:05:37 | INFO     | models.inference:run_inference_batched:361 - Successfully saved complete JSON to /content/drive/MyDrive/vlm-finetuning-project1/results/inference/2b-baseline-p1-test/predictions.json

Base Model Inference Complete! 3004 total samples processed.
Base predictions are secured in: /content/drive/MyDrive/vlm-finetuning-project1/results/inference/2b-baseline-p1-test/predictions.jsonl
